# Direct Tool Calling - Simple Example

This notebook demonstrates **direct tool calling**, where:
1. The model decides to call a tool
2. Your code executes the tool
3. The result goes back to the model's context

Each tool call requires a separate round trip to the model.

In [ ]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

api_key = os.getenv('ANTHROPIC_API_KEY')
client = Anthropic(api_key=api_key)
model = "claude-3-5-sonnet-20241022"

print("✓ Anthropic client initialized")

## Simple Tool: Add Numbers

A very simple tool that just adds two numbers.

In [ ]:
def add_numbers(a: float, b: float) -> float:
    """Add two numbers together."""
    result = a + b
    print(f"Tool executed: add_numbers({a}, {b}) = {result}")
    return result

You define a schema with a name, a description, and an input schema. Claude reads this to decide whether and when to call the tool.

In [ ]:
# Define the tool for Claude
tools = [
    {
        "name": "add_numbers",
        "description": "Add two numbers together and return the result. Use this when the user asks to add, sum, or perform addition on numbers. Do not use this for subtraction, multiplication, division, or other mathematical operations.",
        "input_schema": {
            "type": "object",
            "properties": {
                "a": {
                    "type": "number",
                    "description": "The first number"
                },
                "b": {
                    "type": "number",
                    "description": "The second number"
                }
            },
            "required": ["a", "b"]
        }
    }
]

print("✓ Tool defined")

## Direct Tool Calling Loop

In direct calling, we handle each tool call immediately and send the result back to the model.

- Your code sends a message to Claude including the tool definitions and the user's input.
- Claude issues a tool-use block containing the tool name, a unique ID, and the input arguments it wants to pass. The API response comes back with stop_reason: tool_use.
- Your code executes the tool using those arguments. Note that the assistant turn has already ended (Claude is not holding a connection open or waiting on your server). The model is stateless between calls. To continue, your code makes a fresh API request containing the prior messages plus the tool result.
- You return the result in a tool-result block that references the original tool-use ID.
- Claude continues using the tool result as context for its next response, either another tool-use block or a final end turn.

In [ ]:
def direct_tool_call(user_message: str):
    """Call tool directly - each result goes back to the model's context."""
    messages = [{"role": "user", "content": user_message}]
    
    print(f"\n📤 User: {user_message}")
    print("\n--- Direct Tool Calling Loop ---")
    
    while True:
        # Send message to Claude
        response = client.messages.create(
            model=model,
            max_tokens=256,
            tools=tools,
            messages=messages
        )
        
        print(f"\n[Stop reason: {response.stop_reason}]")
        
        # Check if Claude wants to call a tool
        if response.stop_reason == "tool_use":
            # Find the tool use block
            tool_use = None
            for block in response.content:
                if block.type == "tool_use":
                    tool_use = block
                    break
            
            if tool_use:
                print(f"\n🔧 Claude calls tool: {tool_use.name}")
                print(f"   Input: {tool_use.input}")
                
                # Execute the tool
                if tool_use.name == "add_numbers":
                    result = add_numbers(tool_use.input["a"], tool_use.input["b"])
                else:
                    result = f"Unknown tool: {tool_use.name}"
                
                # Add Claude's response to messages
                messages.append({
                    "role": "assistant",
                    "content": response.content
                })
                
                # Add tool result back to messages (goes to model's context)
                messages.append({
                    "role": "user",
                    "content": [
                        {
                            "type": "tool_result",
                            "tool_use_id": tool_use.id,
                            "content": str(result)
                        }
                    ]
                })
                print(f"   Result sent back to model's context")
        else:
            # Claude has finished - extract text response
            final_response = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_response += block.text
            
            print(f"\n✅ Claude's final response: {final_response}")
            return final_response

## Test Direct Tool Calling

In [ ]:
# Simple request
direct_tool_call("Please add 15 and 27 for me.")

In [ ]:
# Request that might trigger multiple calls
direct_tool_call("What is 10 + 20 and also 5 + 8?")

## Key Characteristics of Direct Tool Calling

✓ **Model sees all data**: Each tool result enters the model's context window  
✓ **Full control**: You see every tool call and result  
✓ **Simple to understand**: Clear request-response flow  
✗ **More tokens**: Raw data consumes tokens  
✗ **More round trips**: Each tool call requires a separate API call  

**Best for**: Small datasets, when the model needs to reason over all results, debugging tool use